In [1]:
# grab directory root
import sys
sys.path.append("../")
import os

from dinov3.eval.tSNE import extract_embeddings
from dinov3.data.datasets import NCells
from dinov3.models.backbone_loader import load_backbone
from dinov3.eval.simpleKNN import evaluate_simple_knn
from dinov3.eval.per_datasetKNN import evaluate_per_dataset_knn
import torch
import json
from torchvision import transforms


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

[rank0]:[W924 15:28:19.393107620 ProcessGroupNCCL.cpp:4561] [PG ID 0 PG GUID 0 Rank 0]  using GPU 0 to perform barrier as devices used by this process are currently unknown. This can potentially cause a hang if this rank to GPU mapping is incorrect. Specify device_ids in barrier() to force use of a particular device, or call init_process_group() with a device_id.


In [2]:
# defaul transform used in dinov3
def make_transform(resize_size: int | list[int] = 768):
    to_tensor = transforms.ToTensor()
    resize = transforms.Resize((resize_size, resize_size), antialias=True)
    normalize = transforms.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225),
    )
    return transforms.Compose([to_tensor, resize, normalize])

test_dataset = NCells(
    root="/home/students/code/helmholtzSS25/Hannes/dinov3playground/manifest_test_fixed.csv.gz",
    # root="/home/students/code/helmholtzSS25/sai/test_datasets/test_helm_IHC.csv.gz",
    split=NCells.Split.TRAIN,
    transform=make_transform(), 
    target_transform=None,
)
print(f"Test Dataset contains {len(test_dataset)} entries")


# cell_labels
import gzip
import pandas as pd

# Directly read into pandas
datasset_origin = pd.read_csv("/home/students/code/helmholtzSS25/Hannes/dinov3playground/manifest_test_fixed.csv.gz", compression="gzip")
datasset_origin['label'] = datasset_origin.apply(
    lambda row: row['origin'] if pd.isna(row['label']) or row['label'] == '' else row['label'],
    axis=1
)
label_to_origin = dict(zip(datasset_origin["label"], datasset_origin["origin"]))

Test Dataset contains 359426 entries


In [3]:
model_paths = {
    # "dinov3-double-teacher-lambda-0-3": {
    #     "config_file":"/home/students/code/helmholtzSS25/sai/rerun_exp/dinov3playground/outputs-diniov3-double-teacher-lambda-slow-annealing-0.2/config.yaml",
    #     "pretrained_weights":"/home/students/code/helmholtzSS25/sai/rerun_exp/dinov3playground/outputs-diniov3-double-teacher-lambda-slow-annealing-0.2/ckpt/123749",
    #     "output_dir":"/home/students/code/helmholtzSS25/Hannes/dinov3playground/CLUSTER_BACKBONE_STUDENT_v3d5_coshdb_margin2"
    # },
    "dinov3-hdb-scan": {
        "config_file":"/home/students/code/helmholtzSS25/Hannes/dinov3playground/CLUSTER_BACKBONE_STUDENT_v3d5_coshdb_margin2/config.yaml",
        "pretrained_weights":"/home/students/code/helmholtzSS25/Hannes/dinov3playground/CLUSTER_BACKBONE_STUDENT_v3d5_coshdb_margin2/ckpt/22499",
        "output_dir":"/home/students/code/helmholtzSS25/Hannes/dinov3playground/CLUSTER_BACKBONE_STUDENT_v3d5_coshdb_margin2"
    },   
    "dinov3-pure-db-scan": {
        "config_file":"/home/students/code/helmholtzSS25/Hannes/dinov3playground/outputs-finetune-dbscan/config.yaml",
        "pretrained_weights":"/home/students/code/helmholtzSS25/Hannes/dinov3playground/outputs-finetune-dbscan/ckpt/22499",
        "output_dir":"/home/students/code/helmholtzSS25/Hannes/dinov3playground/outputs-finetune-dbscan"
    }
}

In [4]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import precision_score, recall_score
import numpy as np
for model_name in model_paths:
    model = load_backbone(
        config_file=model_paths[model_name]['config_file'],
        pretrained_weights=model_paths[model_name]['pretrained_weights'],
        output_dir=model_paths[model_name]['output_dir']
    )
    
    
    target_size = 224
    batch_size = 64
    num_workers = 6
    embeddings, labels = extract_embeddings(
        model=model,
        dataset=test_dataset,
        device=DEVICE,
        batch_size=batch_size,
        num_workers=num_workers,
        target_size=target_size,
    )
    print(np.unique(labels))
    
    dataset_list = [label_to_origin[item] for item in labels]
    
    # knn = KNeighborsClassifier(n_neighbors=5, metric='cosine')
    # knn.fit(embeddings, labels)
    # accuracy_global = knn.score(embeddings, labels)
    
    # pred_labels = knn.predict(embeddings)
    # precision_global = precision_score(labels, pred_labels, average='macro')
    # recall_global = recall_score(labels, pred_labels, average='macro')
    # global_results = {"accuracy":accuracy_global,"precision":precision_global,"recall":recall_global}
    
    # unique_datasets = np.unique(dataset_list)
    # dataset_accuracies = {}

    # for ds in unique_datasets:
    #     idx = [i for i, d in enumerate(dataset_list) if d == ds]
    #     ds_embeddings = embeddings[idx]
    #     ds_labels = np.array(labels)[idx]
    #     ds_accuracy = knn.score(ds_embeddings, ds_labels)
    #     dataset_accuracies[ds] = ds_accuracy    
    full_results = evaluate_per_dataset_knn(embeddings, labels, k_list=[1,3,5,9], metric='cosine', sample_size=None, dataset_ids=dataset_list)
    global_results = full_results['global']
    per_dataset_results = full_results['per_dataset']
    
    with open(f"{model_name}_global_results.json", "w") as json_file:
        json.dump(global_results, json_file, indent=4)

    with open(f"{model_name}_per_dataset_results.json", "w") as json_file:
        json.dump(per_dataset_results, json_file, indent=4)


Materializing model parameters on cuda
Model loaded on 22500 start iteration
Model backbone architecture: 
 DinoVisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 384, kernel_size=(16, 16), stride=(16, 16))
    (norm): Identity()
  )
  (rope_embed): RopePositionEmbedding()
  (blocks): ModuleList(
    (0-11): 12 x SelfAttentionBlock(
      (norm1): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
      (attn): SelfAttention(
        (qkv): Linear(in_features=384, out_features=1152, bias=True)
        (attn_drop): Dropout(p=0.0, inplace=False)
        (proj): Linear(in_features=384, out_features=384, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): LayerScale()
      (norm2): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=384, out_features=1536, bias=True)
        (act): GELU(approximate='none')
        (fc2): Linear(in_features=1536, out_features=384, bias=True)
        (

batches:   0%|          | 25/5617 [00:02<10:43,  8.69it/s] 


KeyboardInterrupt: 